In [1]:
from datasets import VerificationMode, load_dataset
from torch.utils.data import DataLoader
from lightning import LightningDataModule
from typing import Literal

class OppQADataModule(LightningDataModule):
    KEY = "dasyd/OppQA"

    def __init__(
        self,
        batch_size: int = 32,
        task: Literal["binary", "multi", "open","count"] = "multi",
    ):
        super().__init__()

        self.batch_size = batch_size
        self.task = task

    def _load_dataset_split(self, splits: list[str]):
        """Workaround to overcome the missing hf implementation of only dowloading the split shards"""

        return load_dataset(
            OppQADataModule.KEY,
            self.task,
            data_dir=self.task,
            data_files={split: f"{split}-*" for split in splits},
            verification_mode=VerificationMode.NO_CHECKS,
            num_proc=len(splits),
        )

    def prepare_data(self) -> None:
        self._load_dataset_split(["val", "train"])

    def setup(self, stage: str) -> None:
        if stage == "fit":
            self.dataset = self._load_dataset_split(["train", "val"])
        elif stage == "test":
            self.dataset = self._load_dataset_split(["test"])

        self.dataset = self.dataset.with_format("torch")

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.dataset["train"], batch_size=self.batch_size, shuffle=True
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(self.dataset["val"], batch_size=self.batch_size)

    def test_dataloader(self) -> DataLoader:
        return DataLoader(self.dataset["test"], batch_size=self.batch_size)


module = OppQADataModule(batch_size=4, task="count")
module.prepare_data()
module.setup("fit")
# module.setup("test")

loader = module.train_dataloader()
batch = next(iter(loader))

list(batch.keys())
# batch


Resolving data files:   0%|          | 0/37 [00:00<?, ?it/s]

Generating val split:   0%|          | 0/3631 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/32254 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/37 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/37 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/37 [00:00<?, ?it/s]

['question_id',
 'trajectory',
 'question_type',
 'question',
 'answer_type',
 'answer']

In [14]:
from functools import reduce


q = module.dataset["train"].map(lambda x: {"i" : x["answer"]})
# dist = reduce(increase_count, , {})

# for b in module.train_dataloader():
# dist




Map:   0%|          | 0/32254 [00:00<?, ? examples/s]

In [16]:
def increase_count(x, k):
    y = k["i"].item()
    if y not in x:
        x[y] = 0
    x[y] += 1
    return x
dist = reduce(increase_count, q, {})
dist

{0: 15101, 1: 16947, 2: 203, 3: 3}

In [15]:
dist

{tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0): 1,
 tensor(0)

In [9]:
from functools import reduce

def increase_count(x, y):
    if y not in x:
        x[y] = 0
    x[y] += 1
    return x


res = reduce(increase_count, [1,2,3,5,2], {})



res

{1: 1, 2: 2, 3: 1, 5: 1}